# GO-OSC / VASH — Paper Figures

This notebook generates the **arXiv paper figures**:

- **Fig 1** Motivation: energy vs geometric sensitivity under phase-only degradation  
- **Fig 2** Representation geometry: latent trajectory proxy (analytic signal)  
- **Fig 3** Ablation: canonicalization proxy vs drifting gauge vs energy baseline  
- **Fig 4** Data efficiency: VASH linear probe vs energy features  
- **Fig 5 (stress test)** *Reviewer-killer*: robustness to **nuisance amplitude shocks & impulsive noise** (false positives)

All figures are exported as **vector PDF** (for LaTeX) plus **PNG** fallback into `paper_figures/`.


In [108]:
# ===== 1) Imports =====
import os
import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import hilbert

# Your two code modules (place them in the same folder as this notebook, or add to PYTHONPATH)
import vash_indicators_paper as vash  # indicators / HealthIndex
import oscdyn_degradation_benchmark_mathdoc_v4 as oscdyn  # GO-OSC SSM / benchmark utilities

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("Loaded:", vash.__name__, "and", oscdyn.__name__)


Loaded: vash_indicators_paper and oscdyn_degradation_benchmark_mathdoc_v4


In [109]:
# ===== 2) Plot style + LaTeX-friendly export =====
FIG_DIR = Path("paper_figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 2.0,
})

def save_figure(fig, name):
    pdf_path = FIG_DIR / f"{name}.pdf"
    png_path = FIG_DIR / f"{name}.png"
    fig.tight_layout()
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    print("Saved:", pdf_path, "and", png_path)


In [110]:
# ===== 3) Core simulation + windowing helpers =====

def synth_oscillator(
    T:int,
    fs:float,
    f0:float,
    amp:float=1.0,
    phase_jitter:float=0.0,
    freq_wander:float=0.0,
    noise:float=0.05,
    amp_shock_prob:float=0.0,
    amp_shock_scale:float=0.0,
    impulsive_prob:float=0.0,
    impulsive_scale:float=0.0,
    seed:int=0,
):
    """Simple oscillator with optional degradation + nuisance.
    - phase_jitter: random-walk phase noise strength
    - freq_wander: random-walk frequency drift strength (Hz per step, scaled)
    - amp_shock_*: multiplicative amplitude shocks (nuisance)
    - impulsive_*: additive impulsive spikes (nuisance)
    """
    rng = np.random.default_rng(seed)
    t = np.arange(T) / fs
    # frequency random walk
    df = rng.normal(0.0, freq_wander, size=T)
    f = f0 + np.cumsum(df) / max(1, T) * fs  # mild drift
    # phase random walk
    dphi = 2*np.pi*f/fs + rng.normal(0.0, phase_jitter, size=T)
    phi = np.cumsum(dphi)
    y = amp * np.sin(phi)

    # multiplicative amplitude shocks (nuisance)
    if amp_shock_prob > 0 and amp_shock_scale > 0:
        shocks = rng.random(T) < amp_shock_prob
        shock_vals = 1.0 + amp_shock_scale * rng.standard_normal(T)
        shock_vals = np.clip(shock_vals, 0.2, 5.0)
        y = y * np.where(shocks, shock_vals, 1.0)

    # additive gaussian noise
    y = y + noise * rng.standard_normal(T)

    # impulsive spikes (nuisance)
    if impulsive_prob > 0 and impulsive_scale > 0:
        imp = rng.random(T) < impulsive_prob
        spikes = impulsive_scale * rng.standard_normal(T)
        y = y + np.where(imp, spikes, 0.0)

    return y.astype(np.float64)

def window_signal(y: np.ndarray, window: int, hop: int) -> np.ndarray:
    """Slice 1D signal into (n_windows, window) with stride=hop."""
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    T = y.size
    if T < window:
        return np.empty((0, window), dtype=np.float64)
    idx = np.arange(0, T - window + 1, hop, dtype=int)
    if idx.size == 0:
        return np.empty((0, window), dtype=np.float64)
    W = np.stack([y[i:i+window] for i in idx], axis=0)
    return W

def fit_sinusoid_recon_error(x: np.ndarray, fs: float, f_est: float) -> float:
    """Geo proxy: reconstruction error after least-squares fit of sin/cos at f_est."""
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    n = x.size
    t = np.arange(n) / fs
    w = 2*np.pi*f_est
    # least squares
    Phi = np.column_stack([np.sin(w*t), np.cos(w*t)])
    coef, *_ = np.linalg.lstsq(Phi, x, rcond=None)
    xhat = Phi @ coef
    return float(np.mean((x - xhat)**2))

def goosc_like_window_feats(x: np.ndarray, fs: float, prev: dict | None):
    """Extract GO-OSC-like per-window quantities needed by VASH.
    This is a *paper-figure* pipeline: it uses analytic-signal phase to
    proxy GO-OSC phase increments (dphi), omega (frequency), and geo residual.
    """
    z = hilbert(x)
    phase = np.unwrap(np.angle(z))
    dphi = np.diff(phase)
    # omega estimate: median phase increment -> Hz
    omega_hz = float(np.median(dphi) * fs / (2*np.pi))
    omega = np.array([omega_hz], dtype=np.float64)

    geo = fit_sinusoid_recon_error(x, fs, max(1e-3, abs(omega_hz)))
    damp_mean = 0.995  # stable (kept constant for most figures)

    feats = {
        "geo": geo,
        "damp_mean": damp_mean,
        "omega": omega,
        "dphi": dphi,
        # for baselines
        "rms": vash.rms(x),
        "kurtosis": vash.kurtosis(x),
    }
    return feats

def compute_vash_series(windows: np.ndarray, fs: float, baseline_windows: int = 30):
    """Compute VASH indicator time series + simple HI using HealthIndex."""
    feats_list = []
    prev = None
    for w in windows:
        f = goosc_like_window_feats(w, fs, prev)
        feats_list.append(f)
        prev = f

    # Fit baseline on early windows
    hi = vash.HealthIndex(weights={"GSI":1,"PCC":1,"DDI":1,"FWR":1,"MLL":1,"LQF":1})
    hi.fit_baseline(feats_list[:baseline_windows], geo_key="geo", damp_key="damp_mean")

    # Score sequentially to get derived indicators
    out = {k: [] for k in ["GSI","PCC","DDI","FWR","MLL","LQF","rms","kurtosis","omega_hz","geo"]}
    prevf = None
    for f in feats_list:
        s = hi.score(f, prevf)
        # PCC from phase increments (proxy)
        s["PCC"] = vash.compute_pcc(f["dphi"])
        out["GSI"].append(s.get("GSI", np.nan))
        out["PCC"].append(s.get("PCC", np.nan))
        out["DDI"].append(s.get("DDI", np.nan))
        out["FWR"].append(s.get("FWR", np.nan))
        out["MLL"].append(s.get("MLL", np.nan))
        out["LQF"].append(s.get("LQF", np.nan))
        out["rms"].append(f["rms"])
        out["kurtosis"].append(f["kurtosis"])
        out["omega_hz"].append(float(f["omega"][0]))
        out["geo"].append(float(f["geo"]))
        prevf = f

    for k in out:
        out[k] = np.asarray(out[k], dtype=np.float64)
    return out


In [111]:
# ===== 4) Figure builders =====

def fig1_motivation(fs=2000.0, f0=50.0, T=120000, window=2048, hop=512, schedule=(0.0, 0.05, 0.10, 0.15)):
    segT = T // len(schedule)
    ys=[]
    for k,pj in enumerate(schedule):
        ys.append(synth_oscillator(segT, fs, f0, phase_jitter=pj, noise=0.05, seed=100+k))
    y = np.concatenate(ys)
    W = window_signal(y, window, hop)
    S = compute_vash_series(W, fs, baseline_windows=25)

    seg_idx = np.cumsum([0] + [window_signal(ys[k], window, hop).shape[0] for k in range(len(schedule))])

    fig = plt.figure(figsize=(10.5, 3.2))

    ax1 = plt.subplot(1,3,1)
    start = int(0.45*segT)
    end = start + int(0.06*fs)
    tt = np.arange(end-start)/fs
    ax1.plot(tt, y[start:end])
    ax1.set_title("Signal (constant amplitude)")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("y(t)")

    ax2 = plt.subplot(1,3,2)
    ax2.plot(S["rms"])
    for s in seg_idx[1:-1]:
        ax2.axvline(s, linestyle="--", linewidth=1)
    ax2.set_title("Energy metric (RMS)")
    ax2.set_xlabel("Window index")
    ax2.set_ylabel("RMS")

    ax3 = plt.subplot(1,3,3)
    ax3.plot(S["PCC"], label="PCC")
    ax3.plot(S["GSI"], label="GSI")
    for s in seg_idx[1:-1]:
        ax3.axvline(s, linestyle="--", linewidth=1)
    ax3.set_title("Geometric indicators")
    ax3.set_xlabel("Window index")
    ax3.set_ylabel("Indicator value")
    ax3.legend(loc="best", frameon=True)

    return fig

def fig2_representation_geometry(fs=2000.0, f0=50.0, T=120000,
                                 phase_jitter_degraded=0.12,
                                 noise=0.03,
                                 n_points=8000,
                                 edge_trim_frac=0.05,
                                 seed=0):
    """
    Fig 2 (clean): analytic-signal embedding (proxy) with edge trimming and scatter sampling.
    Healthy should look like a thin annulus / ellipse; degraded should be a thicker annulus.
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.signal import hilbert

    rng = np.random.default_rng(seed)

    # --- Generate signals ---
    yH = synth_oscillator(T=T, fs=fs, f0=f0, amp=1.0,
                          phase_jitter=0.0, freq_wander=0.0,
                          noise=noise, seed=7)
    yD = synth_oscillator(T=T, fs=fs, f0=f0, amp=1.0,
                          phase_jitter=phase_jitter_degraded, freq_wander=0.0,
                          noise=noise, seed=8)

    # --- Analytic signal ---
    zH = hilbert(yH)
    zD = hilbert(yD)

    # --- Trim edges to avoid Hilbert boundary artifacts ---
    trim = int(edge_trim_frac * T)
    if trim > 0:
        zH = zH[trim:-trim]
        zD = zD[trim:-trim]

    # --- Robust scaling so both plots share comparable radius ---
    # Using median |z| avoids outlier-driven scaling
    scaleH = np.median(np.abs(zH)) + 1e-12
    scaleD = np.median(np.abs(zD)) + 1e-12
    zH = zH / scaleH
    zD = zD / scaleD

    # --- Random subsample for clean scatter (no polygon artifacts) ---
    def sample(z, n):
        n = min(n, len(z))
        idx = rng.choice(len(z), size=n, replace=False)
        return z[idx]

    zHs = sample(zH, n_points)
    zDs = sample(zD, n_points)

    # --- Shared axis limits (so panels are directly comparable) ---
    rmax = max(np.max(np.abs(zHs)), np.max(np.abs(zDs)))
    lim = 1.05 * rmax

    fig = plt.figure(figsize=(9.5, 3.6))

    ax1 = plt.subplot(1, 2, 1)
    ax1.scatter(np.real(zHs), np.imag(zHs), s=3, alpha=0.1)
    ax1.set_title("Healthy latent trajectory (proxy)")
    ax1.set_xlabel("Re(z)")
    ax1.set_ylabel("Im(z)")
    ax1.set_xlim(-lim, lim); ax1.set_ylim(-lim, lim)
    ax1.set_aspect("equal", adjustable="box")

    ax2 = plt.subplot(1, 2, 2)
    ax2.scatter(np.real(zDs), np.imag(zDs), s=3, alpha=0.1)
    ax2.set_title("Degraded: phase dispersion (proxy)")
    ax2.set_xlabel("Re(z)")
    ax2.set_ylabel("Im(z)")
    ax2.set_xlim(-lim, lim); ax2.set_ylim(-lim, lim)
    ax2.set_aspect("equal", adjustable="box")

    return fig


def _tv(x: np.ndarray) -> float:
    """Total variation (mean absolute step), a simple stability metric."""
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return np.nan
    return float(np.mean(np.abs(np.diff(x))))

def fig3_ablation_canonicalization_errorbars(
    fs=2000.0,
    f0=50.0,
    T=160000,
    window=2048,
    hop=512,
    phase_jitter_degraded=0.10,
    noise=0.06,
    n_seeds=10,
    baseline_windows=25,
    use_score="PCC",   # or "HI" if your compute_vash_series returns it
):
    """
    Fig 3: Canonicalization stabilizes geometric probes (mean ± std over seeds).
    We measure:
      - AUROC for a geometric score (PCC by default)
      - AUROC for a gauge-drift proxy (random circular shift per window)
      - AUROC for RMS energy baseline
    And show a stability metric (TV) as an inset (lower is better).
    """

    auc_can, auc_drift, auc_rms = [], [], []
    tv_can, tv_drift, tv_rms = [], [], []

    rng_master = np.random.default_rng(0)

    for s in range(n_seeds):
        # --- Two-regime signal: healthy then degraded ---
        yH = synth_oscillator(
            T=T//2, fs=fs, f0=f0, amp=1.0,
            phase_jitter=0.0, freq_wander=0.0,
            noise=noise, seed=1000 + 2*s
        )
        yD = synth_oscillator(
            T=T//2, fs=fs, f0=f0, amp=1.0,
            phase_jitter=phase_jitter_degraded, freq_wander=0.0,
            noise=noise, seed=1000 + 2*s + 1
        )
        y = np.concatenate([yH, yD])

        W = window_signal(y, window, hop)  # (nW, window)
        nW = len(W)
        if nW < 30:
            raise ValueError(f"Not enough windows (nW={nW}). Increase T or reduce window/hop.")

        y_true = np.array([0]*(nW//2) + [1]*(nW - nW//2))

        # --- Canonical pipeline (as implemented): compute VASH series on W ---
        S_can = compute_vash_series(W, fs=fs, baseline_windows=baseline_windows)

        # Allow uppercase/lowercase keys
        def get_key(S, k):
            if k in S: return np.asarray(S[k], float)
            kl = k.lower()
            if kl in S: return np.asarray(S[kl], float)
            raise KeyError(f"Missing key {k}. Available: {list(S.keys())}")

        score_can = get_key(S_can, use_score)

        # --- Gauge drift proxy (REAL, no complex ops): random circular shift per window ---
        # This mimics arbitrary window-to-window phase reference, harming comparability.
        rng = np.random.default_rng(2000 + s)
        shifts = rng.integers(low=0, high=window, size=nW)
        W_drift = np.stack([np.roll(W[i], shifts[i]) for i in range(nW)], axis=0)

        S_drift = compute_vash_series(W_drift, fs=fs, baseline_windows=baseline_windows)
        score_drift = get_key(S_drift, use_score)

        # --- RMS baseline ---
        rms = np.sqrt(np.mean(W**2, axis=1))

        # Replace inf with nan then nan->median (simple, consistent)
        def clean(x):
            x = np.asarray(x, float)
            x = np.where(np.isfinite(x), x, np.nan)
            if np.all(np.isnan(x)):
                return np.zeros_like(x)
            med = np.nanmedian(x)
            return np.where(np.isnan(x), med, x)

        score_can = clean(score_can)
        score_drift = clean(score_drift)
        rms = clean(rms)

        # --- AUROC ---
        auc_can.append(roc_auc_score(y_true, score_can))
        auc_drift.append(roc_auc_score(y_true, score_drift))
        auc_rms.append(roc_auc_score(y_true, rms))

        # --- Stability (Total Variation) ---
        tv_can.append(_tv(score_can))
        tv_drift.append(_tv(score_drift))
        tv_rms.append(_tv(rms))

    # --- Plot: mean ± std AUROC + inset stability ---
    means = [np.mean(auc_can), np.mean(auc_drift), np.mean(auc_rms)]
    stds  = [np.std(auc_can),  np.std(auc_drift),  np.std(auc_rms)]

    fig = plt.figure(figsize=(9.5, 3.8))
    ax = plt.subplot(1, 1, 1)

    x = np.arange(3)
    labels = ["GO-OSC+VASH (canonical)", "No canonicalization (gauge drift)", "Energy baseline (RMS)"]
    ax.bar(x, means, yerr=stds, capsize=6)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=10, ha="right")
    ax.set_ylim(0.0, 1.02)
    ax.set_ylabel("AUROC (mean ± std over seeds)")
    ax.set_title("Ablation: Canonicalization Stabilizes Geometric Probes")

    # Inset: stability (lower TV is better)
    ax_in = fig.add_axes([0.64, 0.18, 0.30, 0.30])
    tv_means = [np.mean(tv_can), np.mean(tv_drift), np.mean(tv_rms)]
    tv_stds  = [np.std(tv_can),  np.std(tv_drift),  np.std(tv_rms)]
    ax_in.bar(np.arange(3), tv_means, yerr=tv_stds, capsize=4)
    ax_in.set_title("Stability (total variation ↓)", fontsize=9)
    ax_in.set_xticks([0,1,2])
    ax_in.set_xticklabels(["Can", "Drift", "RMS"], fontsize=8)
    ax_in.tick_params(axis="x", pad=2)
    ax_in.set_facecolor("white")
    ax_in.patch.set_alpha(1.0)
    ax_in.set_position([0.64, 0.22, 0.30, 0.30])
    ax_in.tick_params(axis="y", labelsize=8)
    for spine in ax_in.spines.values():
      spine.set_linewidth(1.0)


    return fig

def fig4_data_efficiency(
    fs=2000.0,
    f0=50.0,
    T=240000,
    window=512,
    hop=128,
    train_sizes=(10, 20, 40, 80, 160, 320, 640),
    baseline_windows=25,
    # --- hardness knobs (Option A) ---
    phase_jitter_deg=0.04,
    freq_wander_deg=0.003,
    noise_level=0.08,
    # --- new: uncertainty estimation ---
    n_seeds=10,
):
    """
    Fig 4: Data efficiency with uncertainty (mean ± std over seeds).
    Compares:
      - Linear probe on VASH indicators (GSI,PCC,DDI,FWR,MLL,LQF)
      - Linear probe on energy features (RMS + FFT peak)
    Shows AUROC vs #labels (log scale) with shaded ±1 std.

    Requires: synth_oscillator, window_signal, compute_vash_series
    """

    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.pipeline import make_pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score

    # --- helpers ---
    key_aliases = {
        "GSI": ["GSI", "gsi"],
        "PCC": ["PCC", "pcc"],
        "DDI": ["DDI", "ddi"],
        "FWR": ["FWR", "fwr"],
        "MLL": ["MLL", "mll"],
        "LQF": ["LQF", "lqf"],
    }

    def get_series(S, name):
        for k in key_aliases[name]:
            if k in S:
                return np.asarray(S[k], dtype=float)
        raise KeyError(f"Missing {name}. Available keys: {list(S.keys())}")

    def clean_matrix(X):
        X = np.asarray(X, dtype=float)
        X = np.where(np.isfinite(X), X, np.nan)
        return X

    # storage: (n_seeds, n_train_sizes)
    auc_vash = np.zeros((n_seeds, len(train_sizes)), dtype=float)
    auc_energy = np.zeros((n_seeds, len(train_sizes)), dtype=float)

    for s in range(n_seeds):
        # --- generate two-regime data (vary seed per replicate) ---
        yH = synth_oscillator(
            T=T // 2, fs=fs, f0=f0, amp=1.0,
            phase_jitter=0.0, freq_wander=0.0,
            noise=noise_level, seed=1000 + 2*s
        )
        yD = synth_oscillator(
            T=T // 2, fs=fs, f0=f0, amp=1.0,
            phase_jitter=phase_jitter_deg, freq_wander=freq_wander_deg,
            noise=noise_level, seed=1000 + 2*s + 1
        )
        y = np.concatenate([yH, yD])

        W = window_signal(y, window, hop)
        nW = len(W)
        if nW < 80:
            raise ValueError(f"Not enough windows for Fig 4: nW={nW}. Increase T or reduce window/hop.")
        y_true = np.array([0] * (nW // 2) + [1] * (nW - nW // 2))

        # --- compute VASH features once per seed ---
        S = compute_vash_series(W, fs=fs, baseline_windows=baseline_windows)
        X_vash = np.column_stack([get_series(S, k) for k in ["GSI", "PCC", "DDI", "FWR", "MLL", "LQF"]])

        # --- energy baseline once per seed ---
        rms = np.sqrt(np.mean(W ** 2, axis=1))
        spec = np.abs(np.fft.rfft(W, axis=1))
        peak = np.max(spec[:, 1:], axis=1)
        X_energy = np.column_stack([rms, peak])

        # --- clean inf->nan ---
        X_vash = clean_matrix(X_vash)
        X_energy = clean_matrix(X_energy)

        # --- fixed train/test split per seed (but randomized) ---
        rng = np.random.default_rng(2000 + s)
        idx = rng.permutation(nW)
        split = int(0.7 * nW)
        tr, te = idx[:split], idx[split:]
        yte = y_true[te]

        # --- models: impute + scale + logistic regression ---
        clf_v = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(max_iter=4000)
        )
        clf_e = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(max_iter=4000)
        )

        # --- for each label budget, RANDOMLY subsample the training set ---
        for j, m in enumerate(train_sizes):
            m = min(int(m), len(tr))
            sub = rng.choice(tr, size=m, replace=False)

            clf_v.fit(X_vash[sub], y_true[sub])
            pv = clf_v.predict_proba(X_vash[te])[:, 1]
            auc_vash[s, j] = roc_auc_score(yte, pv)

            clf_e.fit(X_energy[sub], y_true[sub])
            pe = clf_e.predict_proba(X_energy[te])[:, 1]
            auc_energy[s, j] = roc_auc_score(yte, pe)

    # --- aggregate over seeds ---
    v_mean = auc_vash.mean(axis=0)
    v_std  = auc_vash.std(axis=0)
    e_mean = auc_energy.mean(axis=0)
    e_std  = auc_energy.std(axis=0)

    # --- plot with shaded ± std ---
    fig = plt.figure(figsize=(6.6, 3.8))
    ax = plt.subplot(1, 1, 1)

    x = np.array(train_sizes, dtype=float)

    ax.plot(x, v_mean, marker="o", label="VASH linear probe")
    ax.fill_between(x, v_mean - v_std, v_mean + v_std, alpha=0.18)

    ax.plot(x, e_mean, marker="o", label="Energy features")
    ax.fill_between(x, e_mean - e_std, e_mean + e_std, alpha=0.18)

    ax.axhline(e_mean[-1], ls="--", lw=1.0, alpha=0.4)
    ax.text(
        train_sizes[2],
        e_mean[-1] + 0.03,   # ← was +0.015, bump it up
        "Energy @ 320 labels",
        fontsize=9,
        alpha=0.7
    )
    ax.set_xscale("log")
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("# labeled windows (log scale)")
    ax.set_ylabel("AUROC")
    ax.set_title("Data efficiency: VASH features need fewer labels")
    ax.legend(loc="best", frameon=True)

    return fig




def fig5_stress_test_false_positives(fs=2000.0, f0=50.0, T=160000, window=2048, hop=512):
    """Reviewer-killer stress test:
    Nuisance amplitude shocks + impulsive noise cause energy spikes (false alarms),
    while geometric degradation (phase jitter) is the real target.

    We evaluate detection AUROC for 'phase-degraded' vs 'healthy', while both are
    contaminated with the same nuisance process. Then we show score traces.
    """
    # Healthy with nuisance
    yH = synth_oscillator(T=T//2, fs=fs, f0=f0, amp=1.0,
                      phase_jitter=0.0, freq_wander=0.0,
                      noise=0.05, seed=31)

    yD = synth_oscillator(T=T//2, fs=fs, f0=f0, amp=1.0,
                      phase_jitter=0.10, freq_wander=0.01,
                      noise=0.05, seed=32)

    y=np.concatenate([yH,yD])
    W=window_signal(y, window, hop)
    S=compute_vash_series(W, fs, baseline_windows=25)
    y_true=np.array([0]*(len(W)//2)+[1]*(len(W)-len(W)//2))

    # Simple scalar scores
    score_geo = S["PCC"]  # geometric sensitivity to phase jitter
    score_energy = S["rms"]

    auc_geo = roc_auc_score(y_true, score_geo)
    auc_energy = roc_auc_score(y_true, score_energy)

    fig = plt.figure(figsize=(10.2, 3.8))
    ax1=plt.subplot(1,2,1)
    ax1.plot(score_energy, label=f"RMS (AUROC={auc_energy:.2f})")
    ax1.plot(score_geo, label=f"PCC (AUROC={auc_geo:.2f})")
    ax1.axvline(len(W)//2, linestyle="--", linewidth=1)
    ax1.set_title("Stress test: nuisance shocks induce energy false positives")
    ax1.set_xlabel("Window index")
    ax1.set_ylabel("Score")
    ax1.legend(loc="best", frameon=True)

    ax2=plt.subplot(1,2,2)
    ax2.bar([0,1], [auc_geo, auc_energy])
    ax2.set_xticks([0,1])
    ax2.set_xticklabels(["PCC (geometric)", "RMS (energy)"], rotation=8, ha="right")
    ax2.set_ylim(0,1)
    ax2.set_ylabel("AUROC")
    ax2.set_title("Detection under strong nuisance")

    return fig


In [112]:
# ===== 5) Build all figures (Fig 1–5) and export for LaTeX =====

fig = fig1_motivation(); save_figure(fig, "fig1_motivation"); plt.close(fig)
fig = fig2_representation_geometry(); save_figure(fig, "fig2_representation_geometry"); plt.close(fig)
fig = fig3_ablation_canonicalization_errorbars(use_score="PCC", n_seeds=10)
save_figure(fig, "fig3_ablation_canonicalization"); plt.close(fig)
fig = fig4_data_efficiency(); save_figure(fig, "fig4_data_efficiency"); plt.close(fig)
fig = fig5_stress_test_false_positives(); save_figure(fig, "fig5_stress_test_false_positives"); plt.close(fig)

print("Done. Figures are in:", FIG_DIR.resolve())


Saved: paper_figures/fig1_motivation.pdf and paper_figures/fig1_motivation.png
Saved: paper_figures/fig2_representation_geometry.pdf and paper_figures/fig2_representation_geometry.png


/tmp/ipython-input-3635841997.py:22: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Saved: paper_figures/fig3_ablation_canonicalization.pdf and paper_figures/fig3_ablation_canonicalization.png


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [4]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [4]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [4]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [4]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWar

Saved: paper_figures/fig4_data_efficiency.pdf and paper_figures/fig4_data_efficiency.png
Saved: paper_figures/fig5_stress_test_false_positives.pdf and paper_figures/fig5_stress_test_false_positives.png
Done. Figures are in: /content/paper_figures
